# Notebook 04 — Combined Pipeline & Cross-Signal Correlation

Merge the three per-utterance CSVs produced by notebooks 01–03, compute Pearson
correlations across ASR, LLM, and prosody signals, and visualize as a heatmap.

**Inputs**:
- `output/01_whisper_results.csv`
- `output/02_llm_uncertainty.csv`
- `output/03_prosody_features.csv`

**Outputs**:
- `output/correlation_heatmap.png`
- Printed observations + open questions for the tech report

CPU-only. With n ≈ 20 utterances this is exploratory — no significance claims.


## 1. Load the three CSVs

All CSVs share a `file` column (basename only, e.g. `sample_01.wav`) which is
the join key. Notebook 02's CSV also carries a `transcript` column that would
collide with notebook 01's — we drop the duplicate to keep the merge clean.


In [ ]:
from __future__ import annotations

from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
OUTPUT_DIR = REPO_ROOT / "output"

asr_csv = OUTPUT_DIR / "01_whisper_results.csv"
llm_csv = OUTPUT_DIR / "02_llm_uncertainty.csv"
prosody_csv = OUTPUT_DIR / "03_prosody_features.csv"

for p in (asr_csv, llm_csv, prosody_csv):
    if not p.exists():
        raise FileNotFoundError(
            f"{p.relative_to(REPO_ROOT)} not found. "
            "Run notebooks 01–03 before this one."
        )

df_asr = pd.read_csv(asr_csv)
df_llm = pd.read_csv(llm_csv).drop(columns=["transcript"], errors="ignore")
df_prosody = pd.read_csv(prosody_csv)


## 2. Merge on `file`

Inner join: keep only utterances that have all three signal sources. If an
audio file got transcribed but, say, LLM generation crashed, it's excluded
from the correlation analysis.


In [ ]:
df = df_asr.merge(df_llm, on="file", how="inner").merge(
    df_prosody, on="file", how="inner"
)

print(f"Merged rows: {len(df)}   (asr={len(df_asr)}, llm={len(df_llm)}, prosody={len(df_prosody)})")
df.head()


## 3. Pearson correlation across signals

We pick one representative column per signal type:

| Source | Column | Interpretation |
|---|---|---|
| ASR | `avg_logprob` | higher (less negative) = more confident |
| LLM | `mean_token_entropy` | higher = more uncertain per-token |
| LLM | `semantic_entropy` | higher = more semantically diverse sampled answers |
| Prosody | `f0_std_hz` | higher = more pitch variation |
| Prosody | `intensity_std_db` | higher = more loudness variation |
| Prosody | `pause_ratio` | higher = more silence |
| Prosody | `speaking_rate` | higher = faster delivery |


In [ ]:
signal_cols: list[str] = [
    "avg_logprob",
    "mean_token_entropy",
    "semantic_entropy",
    "f0_std_hz",
    "intensity_std_db",
    "pause_ratio",
    "speaking_rate",
]

corr = df[signal_cols].corr(method="pearson")
corr.round(3)


## 4. Heatmap

Diverging colormap centered at 0 so positive (red) and negative (blue)
correlations read at a glance. Saved to `output/correlation_heatmap.png` and
referenced from the README.


In [ ]:
fig, ax = plt.subplots(figsize=(9, 7))
sns.heatmap(
    corr,
    annot=True,
    cmap="coolwarm",
    center=0,
    vmin=-1,
    vmax=1,
    fmt=".2f",
    square=True,
    ax=ax,
)
ax.set_title(f"Cross-signal correlation: ASR x LLM x Prosody (n={len(df)})")
fig.tight_layout()

png_path = OUTPUT_DIR / "correlation_heatmap.png"
fig.savefig(png_path, dpi=120)
plt.show()

print(f"Saved heatmap to {png_path.relative_to(REPO_ROOT)}")


## 5. Observations and open questions

Auto-derive the strongest cross-source correlation pair from the data (no
invented values), then list open questions for the tech report. At n ≈ 20 we
are **not** running significance tests — these are seeds for follow-up work,
not claims.


In [ ]:
SOURCE = {
    "avg_logprob": "ASR",
    "mean_token_entropy": "LLM",
    "semantic_entropy": "LLM",
    "f0_std_hz": "Prosody",
    "intensity_std_db": "Prosody",
    "pause_ratio": "Prosody",
    "speaking_rate": "Prosody",
}

pairs: list[tuple[str, str, float]] = []
for i, a in enumerate(signal_cols):
    for b in signal_cols[i + 1:]:
        if SOURCE[a] != SOURCE[b]:
            pairs.append((a, b, corr.loc[a, b]))

pairs_sorted = sorted(pairs, key=lambda x: abs(x[2]), reverse=True)

print(f"=== Observations (n = {len(df)}, exploratory) ===\n")
print("Top cross-source Pearson correlations (|r|):")
for a, b, r in pairs_sorted[:5]:
    print(f"  {a:22s} ({SOURCE[a]:<7s}) <-> {b:22s} ({SOURCE[b]:<7s})  r = {r:+.3f}")

print("\n=== Open questions for the tech report ===")
print("- Does low ASR confidence (avg_logprob) correlate with high LLM uncertainty?")
print("- Do prosodic cues (long pauses, high F0 variability) align with any LLM signal?")
print("- Are the three signal sources largely independent (good - multi-source)")
print("  or redundant (LLM already captures what prosody would tell us)?")
print("- Would the ranking change with a larger, labeled sample?")
